In [1]:
import random
from math import log2
from module.conf import PROJECT_DIR
import xgboost as xgb

In [2]:
# Read CSV
def load_csv(filename):
    data = []
    with open(filename, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        for line in lines[1:]:  # Ignore header
            category, message = line.strip().split(',', 1)
            message = message.strip('"')  # remove ()
            data.append([category, message])
    return data

### 1. load data

In [3]:
data = load_csv(filename=f"{PROJECT_DIR}/data/basic/email/spam.csv")
data[:5]

[['ham',
  'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'],
 ['ham', 'Ok lar... Joking wif u oni...'],
 ['spam',
  "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"],
 ['ham', 'U dun say so early hor... U c already then say...'],
 ['ham', "Nah I don't think he goes to usf, he lives around here though"]]

In [4]:
# Extract keywords in msg
def extract_keywords(data, top_n=20):
    word_freq = {}
    for _, message in data:
        words = message.lower() \
            .replace(',', ' ') \
            .replace('.', ' ') \
            .replace('!', ' ') \
            .replace('(', '').replace(')', '') \
            .replace('[', '').replace(']', '') \
            .split()
        for word in words:
            word_freq[word] = word_freq.get(word, 0) + 1

    # sort by frequency and get top N
    sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
    keywords = [word for word, _ in sorted_words[:top_n]]
    return keywords


# Extract features from msg
def extract_features(message, keywords):
    message = message.lower()
    features = []
    for keyword in keywords:
        count = message.count(keyword)
        features.append(count)
    return features


# Change data to number
def preprocess_data(data, keywords):
    processed_data = []
    for category, message in data:
        features = extract_features(message, keywords)
        label = 1 if category == 'spam' else 0  # spam: 1, ham: 0
        processed_data.append(features + [label])
    return processed_data

In [5]:
test_keywords = extract_keywords(data, 200)
# prepared_data = preprocess_data(data, test_keywords)

In [6]:
def train_test_split(data, test_size=0.2):
    n = len(data)
    n_test = int(n * test_size)
    test_indices = set(random.sample(range(n), n_test))
    train_data = [data[i] for i in range(n) if i not in test_indices]
    test_data = [data[i] for i in range(n) if i in test_indices]
    return train_data, test_data

In [ ]:
def main():
    filename = f"{PROJECT_DIR}/data/basic/email/spam.csv"
    raw_data = load_csv(filename)
    print(f"Loaded {len(raw_data)} samples from {filename}")

    keywords = extract_keywords(raw_data, top_n=2000)  # get top 200 keywords
    print(f"Feature keywords: {keywords}")

    data = preprocess_data(raw_data, keywords)

    train_data, test_data = train_test_split(data, test_size=0.2)
    print(f"Train set: {len(train_data)} samples")
    print(f"Test set: {len(test_data)} samples")

    # max_depth = 5
    # tree = build_tree(train_data, max_depth=max_depth)

    # print("\nDecision Tree structure:")
    # print_tree(keywords, tree)

    X_train, y_train =  [row[:-1] for row in train_data], [row[-1] for row in train_data]
    X_test, y_test =  [row[:-1] for row in test_data], [row[-1] for row in test_data]

    model = xgb.XGBClassifier(objective='binary:logistic', max_depth=5, learning_rate=0.1, n_estimators=100)
    model.fit(X_train, y_train)

    correct = 0
    for sample, label in zip(X_test, y_test):
        # pred = predict(tree, sample[:-1])
        pred = model.predict([sample])
        if pred == label:
            correct += 1
    accuracy = correct / len(test_data)
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

    # predict a new message:
    new_message = "Free tickets to win a prize! Call now!"
    new_sample = extract_features(new_message, keywords)
    # prediction = predict(tree, new_sample)
    # print(f"Predict new message: '{new_message}': {'spam' if prediction == 1 else 'ham'}")
    return


# if __name__ == "__main__":
main()

Loaded 5574 samples from c:\Users\nguyenqh\training\python\learn-python/data/basic/email/spam.csv
Feature keywords: ['i', 'to', 'you', 'a', 'the', 'u', 'and', 'is', 'in', 'me', 'my', 'for', 'your', 'it', 'of', 'call', 'have', 'on', 'that', '2', 'are', 'now', 'so', 'but', 'not', 'or', 'can', 'at', 'do', 'will', "i'm", 'ur', 'be', 'if', 'get', 'with', 'just', 'we', 'this', 'no', 'up', 'when', 'from', '4', 'go', '&lt;#&gt;', 'ok', 'free', 'all', 'how', 'out', 'what', 'know', 'like', 'then', 'good', 'got', 'was', 'come', 'am', 'its', 'love', 'time', 'only', '?', 'day', 'send', 'he', 'there', 'want', 'text', 'as', 'by', 'one', "i'll", 'need', 'ü', 'home', 'going', 'about', 'lor', 'sorry', 'see', 'still', 'txt', 'r', 'n', 'reply', 'dont', 'back', 'our', 'she', 'stop', "don't", 'tell', 'mobile', 'new', 'take', 'hi', 'da', 'any', 'today', 'please', 'pls', 'think', 'been', 'they', 'her', 'later', 'k', 'did', 'dear', 'phone', 'some', 'has', 'well', 'great', 'an', 'hey', 'here', 'claim', 'hope', 